In [ ]:
import requests
import pandas as pd
import time

# --- CONFIGURATION ---
API_KEY = ""  # <--- PASTE KEY HERE
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "https://api.openaq.org/v3"

def get_hcmc_locations_robust():
    """
    Fetches ALL Vietnam stations and filters for HCMC client-side
    to avoid Server 500 errors with coordinate search.
    """
    print("Fetching all locations for Vietnam (ISO: VN)...")
    url = f"{BASE_URL}/locations"
    
    # We ask for Vietnam specifically to narrow it down
    params = {
        "iso": "VN",
        "limit": 1000
    }
    
    try:
        response = requests.get(url, headers=HEADERS, params=params)
        
        # Debugging: Print error if it fails again
        if response.status_code != 200:
            print(f"API Error {response.status_code}: {response.text}")
            return []
            
        results = response.json()['results']
        print(f"Found {len(results)} total stations in Vietnam.")
        
        # Filter for Ho Chi Minh City matches in the name or locality
        hcmc_stations = []
        search_terms = ["Ho Chi Minh", "HCMC", "US Consulate"]
        
        for loc in results:
            # Check name and locality fields safely
            name = loc.get('name', '').lower()
            locality = loc.get('locality', '').lower() if loc.get('locality') else ""
            
            if any(term.lower() in name or term.lower() in locality for term in search_terms):
                hcmc_stations.append(loc)
                
        return hcmc_stations

    except Exception as e:
        print(f"Connection Error: {e}")
        return []

def get_measurements(sensor_id, total_pages=5):
    all_data = []
    for page in range(1, total_pages + 1):
        print(f"  > Fetching page {page} for Sensor {sensor_id}...")
        url = f"{BASE_URL}/sensors/{sensor_id}/measurements"
        params = {"limit": 1000, "page": page}
        
        try:
            r = requests.get(url, headers=HEADERS, params=params)
            if r.status_code != 200:
                break
            data = r.json().get('results', [])
            if not data:
                break
            all_data.extend(data)
            time.sleep(0.2) # Rate limit safety
        except:
            break
    return pd.DataFrame(all_data)

# --- MAIN EXECUTION ---
print("1. Searching for stations...")
locations = get_hcmc_locations_robust()

if locations:
    print(f"\nSuccess! Found {len(locations)} stations in HCMC.")
    
    # Summary Table
    summary = []
    for loc in locations:
        for sensor in loc.get('sensors', []):
            summary.append({
                "Station Name": loc['name'],
                "Station ID": loc['id'],
                "Sensor ID": sensor['id'],
                "Parameter": sensor['parameter']['name'],
                "Provider": loc['provider']['name'] if 'provider' in loc else "N/A"
            })
    
    df_summary = pd.DataFrame(summary)
    if not df_summary.empty:
        # Show US Consulate first if available (High Quality Data)
        df_summary['is_consulate'] = df_summary['Station Name'].str.contains("Consulate", case=False)
        df_summary = df_summary.sort_values(by='is_consulate', ascending=False).drop(columns='is_consulate')
        
        print("\n--- Available Sensors (Top 5) ---")
        print(df_summary[['Station Name', 'Sensor ID', 'Parameter']].head(5).to_string(index=False))

        # Auto-download the first one
        target = df_summary.iloc[0]
        print(f"\n2. Downloading data for: {target['Station Name']} ({target['Parameter']})")
        
        df_data = get_measurements(target['Sensor ID'], total_pages=5)
        
        if not df_data.empty:
            fname = f"hcmc_air_quality_{target['Sensor ID']}.csv"
            df_data.to_csv(fname, index=False)
            print(f"Done! Saved to {fname}")
        else:
            print("No measurements found for this sensor.")
    else:
        print("Locations found, but no sensors listed inside them.")
else:
    print("No stations found. Ensure your API key is correct.")

1. Searching for stations...
Fetching all locations for Vietnam (ISO: VN)...
Found 53 total stations in Vietnam.

Success! Found 2 stations in HCMC.

--- Available Sensors (Top 5) ---
                        Station Name  Sensor ID Parameter
US Diplomatic Post: Ho Chi Minh City       4681      pm25
US Diplomatic Post: Ho Chi Minh City      21631      pm25

2. Downloading data for: US Diplomatic Post: Ho Chi Minh City (pm25)
  > Fetching page 1 for Sensor 4681...
  > Fetching page 2 for Sensor 4681...
  > Fetching page 3 for Sensor 4681...
  > Fetching page 4 for Sensor 4681...
  > Fetching page 5 for Sensor 4681...
Done! Saved to hcmc_air_quality_4681.csv


In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

# --- CONFIGURATION ---
API_KEY = ""  # <--- PASTE KEY HERE
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "https://api.openaq.org/v3"


# We use the other sensor ID you found, which likely has newer data
SENSOR_ID = 11357424  
START_DATE = "2024-11-19"
END_DATE = datetime.now().strftime("%Y-%m-%d")

def get_recent_data(sensor_id, start_date, end_date):
    print(f"Fetching data for Sensor {sensor_id} from {start_date} to {end_date}...")
    
    all_data = []
    page = 1
    
    while True:
        url = f"{BASE_URL}/sensors/{sensor_id}/measurements"
        params = {
            "date_from": start_date,
            "date_to": end_date,
            "limit": 1000,
            "page": page,
            "sort": "asc" # Get Jan 2023 first, then Feb 2023, etc.
        }
        
        try:
            r = requests.get(url, headers=HEADERS, params=params)
            if r.status_code != 200:
                print(f"Stop: API Status {r.status_code}")
                break
            
            data = r.json().get('results', [])
            if not data:
                print("Download complete (no more data).")
                break
                
            all_data.extend(data)
            print(f"  > Page {page}: Fetched {len(data)} records (Total: {len(all_data)})")
            
            # Rate limit safety
            time.sleep(0.2)
            page += 1
            
        except Exception as e:
            print(f"Error: {e}")
            break
            
    return pd.DataFrame(all_data)

# --- EXECUTION ---
df = get_recent_data(SENSOR_ID, START_DATE, END_DATE)

if not df.empty:
    # Clean up columns for ML
    if 'period' in df.columns:
        # Extract the 'start' time from the period dictionary if nested
        # OpenAQ v3 often returns period as a dict: {'start': '...', 'end': '...'}
        df['datetime'] = df['period'].apply(lambda x: x.get('datetimeFrom', {}).get('utc') if isinstance(x, dict) else x)
    
    filename = f"hcmc_cmt8_pm25_{START_DATE}_to_{END_DATE}.csv"
    df.to_csv(filename, index=False)
    print(f"\nSUCCESS! Saved {len(df)} recent records to {filename}")
    print("Columns:", df.columns.tolist())
else:
    print("No data found. Try changing the Sensor ID back to 4681 if 21631 fails.")

Fetching data for Sensor 11357424 from 2024-11-19 to 2026-01-18...
  > Page 1: Fetched 1000 records (Total: 1000)
  > Page 2: Fetched 1000 records (Total: 2000)
  > Page 3: Fetched 1000 records (Total: 3000)
  > Page 4: Fetched 1000 records (Total: 4000)
  > Page 5: Fetched 1000 records (Total: 5000)
  > Page 6: Fetched 1000 records (Total: 6000)
  > Page 7: Fetched 1000 records (Total: 7000)
  > Page 8: Fetched 1000 records (Total: 8000)
  > Page 9: Fetched 878 records (Total: 8878)
Download complete (no more data).

SUCCESS! Saved 8878 recent records to hcmc_cmt8_pm25_2024-11-19_to_2026-01-18.csv
Columns: ['value', 'flagInfo', 'parameter', 'period', 'coordinates', 'summary', 'coverage', 'datetime']


In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

# --- CONFIGURATION ---
API_KEY = ""  # <--- PASTE KEY HERE
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "https://api.openaq.org/v3"

def find_working_sensor():
    print("1. Scanning for ACTIVE sensors in Vietnam (skipping broken ones)...")
    
    # We fetch ALL Vietnam stations to avoid the "radius" error
    url = f"{BASE_URL}/locations"
    params = {
        "iso": "VN",
        "limit": 1000,
        "measurements_only": "true"
    }
    
    try:
        r = requests.get(url, headers=HEADERS, params=params)
        if r.status_code != 200:
            print(f"API Error: {r.text}")
            return None
            
        locations = r.json().get('results', [])
        print(f"   > Found {len(locations)} total stations in Vietnam.")
        
        # Filter: Must be in HCMC + Have data in 2025/2026
        candidates = []
        for loc in locations:
            # Check if it's in Ho Chi Minh City
            name_check = "Ho Chi Minh" in loc.get('name', '') or "HCMC" in loc.get('name', '')
            # Some locations hide city in 'locality', check that too
            locality_check = False
            if loc.get('locality'):
                locality_check = "Ho Chi Minh" in loc.get('locality', '')
            
            if name_check or locality_check:
                for sensor in loc.get('sensors', []):
                    # We only want PM2.5
                    if sensor['parameter']['name'] == 'pm25':
                        # Check dates: specific sensor object has 'datetimeLast'
                        # But in the list view, we might rely on the location's last update
                        # Let's verify specifically.
                        last_update = loc.get('datetimeLast', {}).get('utc', '2000-01-01')
                        
                        if '2025' in last_update or '2026' in last_update:
                            candidates.append({
                                'id': sensor['id'],
                                'station': loc['name'],
                                'last_seen': last_update
                            })
        
        if not candidates:
            print("   > No active 2025/2026 sensors found in HCMC.")
            return None
            
        # Sort by most recent first
        candidates.sort(key=lambda x: x['last_seen'], reverse=True)
        best_sensor = candidates[0]
        print(f"   > SUCCESS: Found active sensor! ID {best_sensor['id']} at '{best_sensor['station']}'")
        print(f"   > Last reported: {best_sensor['last_seen']}")
        return best_sensor['id']

    except Exception as e:
        print(f"Scan failed: {e}")
        return None

def download_history(sensor_id):
    print(f"\n2. Downloading history for Sensor {sensor_id}...")
    all_data = []
    page = 1
    
    # We want data from Jan 2020 to Now (2 full years)
    start_date = "2020-01-01T00:00:00Z"
    end_date = datetime.now().strftime("%Y-%m-%dT%H:%M:%SZ")
    
    while True:
        url = f"{BASE_URL}/sensors/{sensor_id}/measurements"
        params = {
            "datetime_from": start_date,
            "datetime_to": end_date,
            "limit": 1000,
            "page": page,
            "sort": "asc" # Oldest to Newest
        }
        
        try:
            r = requests.get(url, headers=HEADERS, params=params)
            
            # Robust JSON handling
            try:
                data = r.json().get('results', [])
            except:
                print(f"   > Error: API returned non-JSON on page {page} (Rate limit or Server Error).")
                break
                
            if not data:
                print("   > Download complete.")
                break
            
            all_data.extend(data)
            print(f"   > Page {page}: Retrieved {len(data)} records...")
            
            time.sleep(0.3) # Be nice to the API
            page += 1
            
        except Exception as e:
            print(f"Error: {e}")
            break
            
    return pd.DataFrame(all_data)

# --- EXECUTION ---
sensor_id = find_working_sensor()

if sensor_id:
    df = download_history(sensor_id)
    
    if not df.empty:
        # Clean timestamps
        if 'period' in df.columns:
            df['datetime'] = df['period'].apply(lambda x: x.get('datetimeFrom', {}).get('utc') if isinstance(x, dict) else x)
            
        filename = f"hcmc_active_data_{sensor_id}.csv"
        df.to_csv(filename, index=False)
        print(f"\nSaved {len(df)} records to {filename}")
        print("You can now proceed to visualization.")
    else:
        print("Sensor was active but returned no historical data for this range.")
else:
    print("Could not find a working sensor. Try expanding search to 'Southern Vietnam'.")

1. Scanning for ACTIVE sensors in Vietnam (skipping broken ones)...
   > Found 53 total stations in Vietnam.
   > SUCCESS: Found active sensor! ID 21631 at 'US Diplomatic Post: Ho Chi Minh City'
   > Last reported: 2025-03-04T12:00:00Z

2. Downloading history for Sensor 21631...
   > Page 1: Retrieved 1000 records...
   > Page 2: Retrieved 1000 records...
   > Page 3: Retrieved 1000 records...
   > Page 4: Retrieved 1000 records...
   > Page 5: Retrieved 1000 records...
   > Page 6: Retrieved 1000 records...
   > Page 7: Retrieved 1000 records...
   > Page 8: Retrieved 1000 records...
   > Page 9: Retrieved 1000 records...
   > Page 10: Retrieved 1000 records...
   > Page 11: Retrieved 1000 records...
   > Page 12: Retrieved 1000 records...
   > Page 13: Retrieved 1000 records...
   > Page 14: Retrieved 1000 records...
   > Page 15: Retrieved 1000 records...
   > Page 16: Retrieved 1000 records...
   > Download complete.

Saved 16000 records to hcmc_active_data_21631.csv
You can now pr

In [ ]:
import requests
import pandas as pd
from math import radians, cos, sin, asin, sqrt

# --- CONFIGURATION ---
API_KEY = "" # <--- PASTE KEY
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "https://api.openaq.org/v3"

# HCMC Center Coordinates
HCMC_LAT = 10.7769
HCMC_LON = 106.7009

def haversine(lon1, lat1, lon2, lat2):
    """Calculate distance in km between two points."""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    return c * 6371 

def find_hidden_sensors():
    print("1. Downloading ALL Vietnam stations...")
    
    url = f"{BASE_URL}/locations"
    params = {"iso": "VN", "limit": 1000, "measurements_only": "true"}
    
    try:
        r = requests.get(url, headers=HEADERS, params=params)
        locations = r.json().get('results', [])
        print(f"   > Downloaded {len(locations)} stations.")
        
        print("\n2. Filtering for HCMC (Distance < 30km)...")
        candidates = []
        
        for loc in locations:
            # 1. Safety Check: Coordinates
            coords = loc.get('coordinates')
            if not coords:
                continue
            
            lat = coords.get('latitude')
            lon = coords.get('longitude')
            if lat is None or lon is None:
                continue
            
            # 2. Calculate Distance
            dist = haversine(HCMC_LON, HCMC_LAT, lon, lat)
            
            if dist <= 30: # 30km Radius
                # 3. Safety Check: Last Seen Date
                # Handle cases where datetimeLast is None
                dt_obj = loc.get('datetimeLast') or {}
                active_date = dt_obj.get('utc', 'N/A')
                
                candidates.append({
                    "id": loc['id'],
                    "name": loc.get('name', 'Unknown'),
                    "distance": f"{dist:.1f} km",
                    "last_seen": active_date
                })

        # Report Results
        if candidates:
            print(f"\n✅ FOUND {len(candidates)} STATIONS IN HCMC:")
            
            df = pd.DataFrame(candidates)
            df = df.sort_values('last_seen', ascending=False)
            
            # Print table
            print(df.to_string(index=False))
            
            # Check for recent data
            recent = [c for c in candidates if "2024" in c['last_seen'] or "2025" in c['last_seen']]
            
            if recent:
                best = recent[0]
                print(f"\n>>> RECOMMENDATION: Sensor ID {best['id']} ({best['name']})")
                return best['id']
            else:
                print("\n⚠️ WARNING: All found stations seem old (pre-2024).")
                print("You may need to rely on the 2020-2025 US Consulate dataset you already built.")
                return None
        else:
            print("No stations found within 30km of HCMC.")
            return None

    except Exception as e:
        print(f"Error: {e}")
        return None

# --- EXECUTION ---
target_id = find_hidden_sensors()

1. Downloading ALL Vietnam stations...
   > Downloaded 53 stations.

2. Filtering for HCMC (Distance < 30km)...

✅ FOUND 10 STATIONS IN HCMC:
     id                                                          name distance            last_seen
 268816                                                       outdoor   9.4 km                  N/A
 268821                                                      outdoor2   9.4 km                  N/A
 268929                                                           od3   9.4 km                  N/A
 268935                                                           od5   7.5 km                  N/A
 268937                                                           od6   9.4 km                  N/A
3276359                                                          CMT8   3.5 km 2026-01-17T09:00:00Z
6068138                                                   Care Centre   4.4 km 2025-12-08T02:00:00Z
4743591 Trường Đại học Khoa học Tự nhiên, ĐHQG-HCM, Cơ sở 

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime

# --- CONFIGURATION ---
API_KEY = ""  # <--- PASTE KEY HERE
HEADERS = {"X-API-Key": API_KEY}
BASE_URL = "https://api.openaq.org/v3"

# We check BOTH known US Consulate sensors to fill the 2020-2023 gap
TARGET_SENSORS = [4681, 21631] 

def download_range(sensor_id, start_year, end_year):
    print(f"\nScanning Sensor {sensor_id} for data ({start_year}-{end_year})...")
    all_data = []
    
    # We loop by YEAR to avoid timeouts and manage large requests
    for year in range(start_year, end_year + 1):
        print(f"  > Checking Year {year}...")
        
        start_date = f"{year}-01-01T00:00:00Z"
        end_date = f"{year}-12-31T23:59:59Z"
        page = 1
        year_data_count = 0
        
        while True:
            url = f"{BASE_URL}/sensors/{sensor_id}/measurements"
            params = {
                "datetime_from": start_date,
                "datetime_to": end_date,
                "limit": 1000,
                "page": page,
                "sort": "asc"
            }
            
            try:
                r = requests.get(url, headers=HEADERS, params=params)
                if r.status_code != 200:
                    break 
                
                data = r.json().get('results', [])
                if not data:
                    break
                
                all_data.extend(data)
                year_data_count += len(data)
                page += 1
                time.sleep(0.2) # Rate limit safety
                
            except Exception as e:
                print(f"    ! Error: {e}")
                break
        
        if year_data_count > 0:
            print(f"    + Found {year_data_count} records for {year}.")
        else:
            print(f"    - No data for {year}.")
            
    return pd.DataFrame(all_data)

# --- EXECUTION ---
print("--- STARTING UNIFIED DOWNLOAD (2020 - Present) ---")
dfs = []

# 1. Download from all candidate sensors
for sensor in TARGET_SENSORS:
    df_sensor = download_range(sensor, 2018, 2022)
    if not df_sensor.empty:
        # Standardize columns
        if 'period' in df_sensor.columns:
            df_sensor['datetime'] = df_sensor['period'].apply(
                lambda x: x.get('datetimeFrom', {}).get('utc') if isinstance(x, dict) else x
            )
        # Add a column to track which sensor gave us this data (useful for debugging)
        df_sensor['source_sensor'] = sensor
        dfs.append(df_sensor)

# 2. Merge and Clean
if dfs:
    full_df = pd.concat(dfs)
    
    # Convert to datetime objects
    full_df['dt'] = pd.to_datetime(full_df['datetime'])
    
    # SORT by time
    full_df = full_df.sort_values('dt')
    
    # REMOVE DUPLICATES
    # If both sensors covered the same hour, keep the one that came first (or average them)
    # Here we simply drop duplicates based on the timestamp
    before_count = len(full_df)
    full_df = full_df.drop_duplicates(subset=['dt'], keep='first')
    after_count = len(full_df)
    
    print(f"\n--- MERGE COMPLETE ---")
    print(f"Total Raw Records: {before_count}")
    print(f"Unique Hourly Records: {after_count}")
    print(f"Time Range: {full_df['dt'].min()} to {full_df['dt'].max()}")
    
    # Save
    filename = "hcmc_full_2018_2022.csv"
    full_df.to_csv(filename, index=False)
    print(f"Saved to: {filename}")
    
else:
    print("No data found from 2018-2022 on these sensors.")

--- STARTING UNIFIED DOWNLOAD (2024 - Present) ---


TypeError: 'int' object is not iterable